<a href="https://colab.research.google.com/github/A7reus/DNA-Sequence-Classifier/blob/main/DSA_Sequence_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.1 Imports + Seed

- `pandas` for tables
- `scikit-learn` for models
- `matplotlib` for data analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import random

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
print("setup ok:", pd.__version__)

setup ok: 2.2.3


# Load Promoter Data

The downloaded data is in format: `+,S10,    sequence` and therefore needs to be parsed.

In [2]:
import urllib.request
from io import StringIO

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"
raw = urllib.request.urlopen(url).read().decode("utf-8")

# Format per UCI: [class, id, sequence]
rows = []
for line in raw.strip().split("\n"):
  parts = line.strip().split(",")
  if len(parts) != 3: continue

  label, name, seq = parts
  seq = seq.strip().lower()
  rows.append([label.strip(), seq])

df = pd.DataFrame(rows, columns=["label", "sequence"])
df["label_num"] = df["label"].map({"+": 1, "-": 0})
df["length"] = df["sequence"].str.len()

print(df.shape)
print(df.head(3).to_string())
print(df["label"].value_counts().to_string())
print("unique lengths:", df["length"].unique())

(106, 4)
  label                                                   sequence  label_num  length
0     +  tactagcaatacgcttgcgttcggtggttaagtatgtataatgcgcgggcttgtcgt          1      57
1     +  tgctatcctgacagttgtcacgctgattggtgtcgttacaatctaacgcatcgccaa          1      57
2     +  gtactagagaactagtgcattagcttatttttttgttatcatgctaaccacccggcg          1      57
label
+    53
-    53
unique lengths: [57]


In [3]:
assert df.shape[0] == 106, "should be 106"
assert set(df["length"].unique()) == {57}, "all 57bp"
assert df["label"].value_counts()["+"] == 53, "+ve should be 53"
assert df["label"].value_counts()["-"] == 53, "-ve should be 53 too"

print("sanity passed")
print(df["sequence"].str[0].value_counts())

sanity passed
sequence
t    38
c    27
a    26
g    15
Name: count, dtype: int64
